# Baseline (Masking)

This notebook extracts the masking-based pipeline from `reactOT.ipynb` and reformats it as a standalone baseline notebook.
It is intended to live in `../generative_chem_reaction_structures/Notebooks/`, so all relative paths follow that directory structure.

**What this notebook does:**
- Load the pickled reaction dataset
- Keep reactions with variable atom counts instead of filtering to a fixed size
- Pad each batch to the largest molecule in that batch
- Build a binary mask indicating which atom slots are real and which are padding
- Train a mask-aware Flow-EGNN with a flow-matching objective
- Evaluate masked RMSD after Euler integration


## Notebook Outline
1. Imports and device setup
2. Load dataset and inspect atom-count variability
3. Build a padded dataset and collate function
4. Construct masked flow-matching batches
5. Define a mask-aware Flow-EGNN
6. Train with a masked loss
7. Evaluate with masked RMSD and masked ODE integration


In [1]:
# Core imports and device selection.
import pickle
import math
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("Using device:", device)


Using device: mps


In [2]:
# Load the pickled dataset.
pkl_path = "../Data/train_rpsb_all.pkl"
with open(pkl_path, "rb") as f:
    obj = pickle.load(f)

print(obj.keys())
print(obj["reactant"].keys())


dict_keys(['reactant', 'transition_state', 'product', 'single_fragment', 'use_ind', 'ts_guess', 'ts_guess_sbv1', 'ts_guess_true', 'ts_guess_NEBCI-xtb'])
dict_keys(['num_atoms', 'charges', 'fragments', 'positions', 'rxn', 'wB97x_6-31G(d).energy', 'wB97x_6-31G(d).atomization_energy', 'wB97x_6-31G(d).forces', 'formula', 'xtb_positions'])


## Variable Atom Counts and Why Masking Is Needed
In the fixed-size baseline, batching is easy because every reaction in the batch has the same number of atoms.
Here we do not throw away reactions with different sizes. Instead, each minibatch is padded up to the largest molecule in that batch.

That creates two types of atom slots:
- Real atoms: positions and atomic numbers that correspond to a real molecule
- Padded atoms: artificial placeholders added only so tensors can share one shape

The mask keeps track of that distinction. A value of `1` means the slot is a real atom and should contribute to message passing, losses, and evaluation. A value of `0` means the slot is padding and should be ignored.


In [3]:
# Inspect how much the atom count varies across reactions.
lengths = np.array([
    obj["reactant"]["positions"][i].shape[0]
    for i in range(len(obj["reactant"]["positions"]))
])

print("N min/median/max:", lengths.min(), np.median(lengths), lengths.max())
print("Number of samples:", len(lengths))
print("Example TS guess shape:", np.array(obj["ts_guess_NEBCI-xtb"][0]).shape)


N min/median/max: 4 14.0 23
Number of samples: 10073
Example TS guess shape: (7, 3)


## Padded Dataset and Collation
Each dataset item returns one reaction with its own atom count `N`.
The collate function then pads a full minibatch to `Nmax = max(N in batch)`.

The important outputs of collation are:
- `xR`, `xP`, `xT`, `x0_guess`: padded coordinate tensors of shape `(B, Nmax, 3)`
- `z`: padded atomic-number tensor of shape `(B, Nmax)`
- `mask`: binary tensor of shape `(B, Nmax)` with ones on real atoms
- `N`: the original unpadded atom counts for reference

This is the first place where masking matters. Padding lets us use dense tensors, and the mask prevents those padded entries from affecting learning.


In [4]:
class RPSBDataset(Dataset):
    """Dataset wrapper for variable-size reactant/product/TS structures."""
    def __init__(self, obj, split_indices=None):
        self.obj = obj
        self.indices = split_indices if split_indices is not None else list(range(len(obj["reactant"]["positions"])))

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, k):
        i = self.indices[k]
        xR = self.obj["reactant"]["positions"][i]
        xP = self.obj["product"]["positions"][i]
        xT = self.obj["transition_state"]["positions"][i]
        x0_guess = self.obj["ts_guess_NEBCI-xtb"][i]
        z = np.array(self.obj["reactant"]["charges"][i], dtype=np.int64)

        return {
            "xR": torch.tensor(xR, dtype=torch.float32),
            "xP": torch.tensor(xP, dtype=torch.float32),
            "xT": torch.tensor(xT, dtype=torch.float32),
            "x0_guess": torch.tensor(x0_guess, dtype=torch.float32),
            "z": torch.tensor(z, dtype=torch.long),
            "N": xR.shape[0],
        }


def collate_pad(batch):
    """Pad a variable-size batch and return a binary mask for real atoms."""
    B = len(batch)
    Ns = [item["N"] for item in batch]
    Nmax = max(Ns)

    xR = torch.zeros(B, Nmax, 3, dtype=torch.float32)
    xP = torch.zeros(B, Nmax, 3, dtype=torch.float32)
    xT = torch.zeros(B, Nmax, 3, dtype=torch.float32)
    x0g = torch.zeros(B, Nmax, 3, dtype=torch.float32)
    z = torch.zeros(B, Nmax, dtype=torch.long)
    mask = torch.zeros(B, Nmax, dtype=torch.float32)

    for b, item in enumerate(batch):
        N = item["N"]
        xR[b, :N] = item["xR"]
        xP[b, :N] = item["xP"]
        xT[b, :N] = item["xT"]
        x0g[b, :N] = item["x0_guess"]
        z[b, :N] = item["z"]
        mask[b, :N] = 1.0

    return {
        "xR": xR,
        "xP": xP,
        "xT": xT,
        "x0_guess": x0g,
        "z": z,
        "mask": mask,
        "N": torch.tensor(Ns),
    }


In [5]:
# Train/validation split and dataloaders.
N_total = len(obj["reactant"]["positions"])
perm = np.random.permutation(N_total)
n_train = int(0.9 * N_total)
train_idx = perm[:n_train].tolist()
val_idx = perm[n_train:].tolist()

train_ds = RPSBDataset(obj, train_idx)
val_ds = RPSBDataset(obj, val_idx)

train_dl = DataLoader(train_ds, batch_size=32, shuffle=True, collate_fn=collate_pad)
val_dl = DataLoader(val_ds, batch_size=32, shuffle=False, collate_fn=collate_pad)

batch = next(iter(train_dl))
for k, v in batch.items():
    if torch.is_tensor(v):
        print(k, tuple(v.shape), v.dtype)
    else:
        print(k, type(v))
print("mask sums per sample:", batch["mask"].sum(dim=1)[:8])
print("train/val sizes:", len(train_ds), len(val_ds))


xR (32, 21, 3) torch.float32
xP (32, 21, 3) torch.float32
xT (32, 21, 3) torch.float32
x0_guess (32, 21, 3) torch.float32
z (32, 21) torch.int64
mask (32, 21) torch.float32
N (32,) torch.int64
mask sums per sample: tensor([11., 12., 12., 14., 17., 17., 21., 13.])
train/val sizes: 9065 1008


## Masked Flow-Matching Batch Construction
We still train a velocity field between a start configuration and the transition-state target.
The difference from the no-masking notebook is that the conditioning dictionary now also carries the mask.

For each sample we define:
- `x0`: the initial structure used as the start of the flow. Here we use the XTB TS guess.
- `x1`: the true transition-state geometry
- `x_t = (1 - t) x0 + t x1`: a point sampled along the straight interpolation path
- `v_star = x1 - x0`: the target constant velocity for conditional flow matching

The mask does not change this interpolation formula. Instead, it tells the model and the loss which coordinates are meaningful.


In [6]:
def make_flow_batch_padded(batch, device):
    """Move a padded batch to device and build flow-matching targets."""
    xR = batch["xR"].to(device)
    xP = batch["xP"].to(device)
    xT = batch["xT"].to(device)
    x0g = batch["x0_guess"].to(device)
    z = batch["z"].to(device)
    mask = batch["mask"].to(device)

    x0 = x0g
    x1 = xT

    B = xR.shape[0]
    t = torch.rand(B, device=device)

    x_t = (1 - t)[:, None, None] * x0 + t[:, None, None] * x1
    v_star = x1 - x0
    cond = {"z": z, "xR_ctx": xR, "xP_ctx": xP, "mask": mask}
    return x_t, t, cond, v_star, mask

x_t, t, cond, v_star, mask = make_flow_batch_padded(batch, device)
print("x_t", x_t.shape, "t", t.shape, "v_star", v_star.shape, "mask", mask.shape)
print("active atoms in first batch item:", int(mask[0].sum().item()))


x_t torch.Size([32, 21, 3]) t torch.Size([32]) v_star torch.Size([32, 21, 3]) mask torch.Size([32, 21])
active atoms in first batch item: 11


## Flow-EGNN Model with Masking
The model architecture is the same basic idea as the ReactOT baseline: atomic-number embeddings, time embeddings, reactant/product context, and EGNN message passing.
The difference is that every stage has to respect padding.

Masking enters in three places:
1. Edge construction: only edges between real atoms are kept
2. Message passing and coordinate updates: messages touching padded atoms are zeroed out
3. Output velocity: padded atoms return zero velocity

This keeps the padded region inert. Without those constraints, the model could waste capacity fitting meaningless padded coordinates and could contaminate real-atom updates.


In [7]:
class TimeEmbedding(nn.Module):
    """Fourier-style time embedding used by the Flow-EGNN."""
    def __init__(self, n_freq=16):
        super().__init__()
        self.register_buffer("freqs", 2 * math.pi * (2 ** torch.arange(n_freq).float()))

    def forward(self, t):
        args = t[:, None] * self.freqs[None, :]
        return torch.cat([torch.sin(args), torch.cos(args)], dim=-1)


def radius_edges_batch(x, node_mask, r_cut=4.5, include_self=False):
    """Build radius-based edges while ignoring padded atoms."""
    B, N, _ = x.shape
    device = x.device

    diff = x[:, :, None, :] - x[:, None, :, :]
    d2 = (diff ** 2).sum(dim=-1)

    valid = (node_mask[:, :, None] * node_mask[:, None, :]) > 0.5
    within = d2 <= (r_cut ** 2)
    adj = valid & within

    if not include_self:
        eye = torch.eye(N, device=device, dtype=torch.bool)[None, :, :]
        adj = adj & (~eye)

    src_list = []
    dst_list = []
    for b in range(B):
        s, d = torch.where(adj[b])
        offset = b * N
        src_list.append(s + offset)
        dst_list.append(d + offset)

    src = torch.cat(src_list, dim=0)
    dst = torch.cat(dst_list, dim=0)
    return src, dst


class EGNNLayerMaskedFlat(nn.Module):
    """Mask-aware EGNN layer over a flattened batch graph."""
    def __init__(self, h_dim):
        super().__init__()
        self.phi_m = nn.Sequential(
            nn.Linear(2 * h_dim + 1, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
        )
        self.phi_h = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, h_dim),
        )
        self.phi_x = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, 1),
        )

    def forward(self, h, x, src, dst, node_mask):
        B, N, H = h.shape
        h_flat = h.reshape(B * N, H)
        x_flat = x.reshape(B * N, 3)
        m_flat = node_mask.reshape(B * N, 1)

        xi = x_flat[src]
        xj = x_flat[dst]
        rij = xi - xj
        dij = (rij ** 2).sum(dim=-1, keepdim=True)

        hi = h_flat[src]
        hj = h_flat[dst]
        mij = self.phi_m(torch.cat([hi, hj, dij], dim=-1))

        e_mask = m_flat[src] * m_flat[dst]
        mij = mij * e_mask

        agg = torch.zeros((B * N, H), device=h.device, dtype=h.dtype)
        agg.index_add_(0, src, mij)

        h_flat = h_flat + self.phi_h(agg)
        h_flat = h_flat * m_flat

        s = self.phi_x(mij)
        dx_ij = rij * s
        dx = torch.zeros((B * N, 3), device=x.device, dtype=x.dtype)
        dx.index_add_(0, src, dx_ij)
        dx = dx * m_flat

        x_flat = x_flat + dx

        h = h_flat.reshape(B, N, H)
        x = x_flat.reshape(B, N, 3)
        return h, x


class FlowEGNN(nn.Module):
    """Mask-aware Flow-EGNN conditioned on reactant/product geometry."""
    def __init__(self, z_max=100, h_dim=128, n_layers=4, time_freq=16, use_concat_ctx=True):
        super().__init__()
        self.use_concat_ctx = use_concat_ctx
        self.z_emb = nn.Embedding(z_max + 1, h_dim)

        self.t_emb = TimeEmbedding(n_freq=time_freq)
        self.t_proj = nn.Linear(2 * time_freq, h_dim)

        if use_concat_ctx:
            self.ctx_proj = nn.Linear(9, h_dim)
        else:
            self.ctx_proj = nn.Linear(3, h_dim)

        self.layers = nn.ModuleList([EGNNLayerMaskedFlat(h_dim) for _ in range(n_layers)])
        self.v_head = nn.Sequential(
            nn.Linear(h_dim, h_dim),
            nn.SiLU(),
            nn.Linear(h_dim, 3),
        )

    def forward(self, x_t, t, cond):
        z = cond["z"]
        xR = cond["xR_ctx"]
        xP = cond["xP_ctx"]
        node_mask = cond["mask"]

        B, N, _ = x_t.shape
        h = self.z_emb(z)

        te = self.t_proj(self.t_emb(t))
        h = h + te[:, None, :].expand(B, N, -1)

        diff = xP - xR
        if self.use_concat_ctx:
            ctx = torch.cat([xR, xP, diff], dim=-1)
        else:
            ctx = diff
        h = h + self.ctx_proj(ctx)

        src, dst = radius_edges_batch(x_t, node_mask, r_cut=4.5)

        x = x_t
        for layer in self.layers:
            h, x = layer(h, x, src, dst, node_mask)

        return self.v_head(h) * node_mask[..., None]


model = FlowEGNN(h_dim=128, n_layers=3, use_concat_ctx=True).to(device)
v_pred = model(x_t, t, cond)
print("v_pred shape:", v_pred.shape)
print("mean |v| on real atoms:", float(v_pred.norm(dim=-1)[mask > 0].mean()))


v_pred shape: torch.Size([32, 21, 3])
mean |v| on real atoms: 0.4735950529575348


/var/folders/8p/6t_fjn5d1c7fxwflz29kdvqr0000gn/T/ipykernel_78838/530462697.py:152: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:837.)
  print("mean |v| on real atoms:", float(v_pred.norm(dim=-1)[mask > 0].mean()))


## Masked Loss and Evaluation
The loss should only penalize errors on real atoms. If padded atoms were included, the objective would be dominated by arbitrary zero-filled entries.

The same rule applies at evaluation time:
- `masked_rmsd` averages only over true atoms
- `euler_solve_masked` zeros out padded velocities and restores padded coordinates after each step

That second detail is important. Even if the model returns nearly zero velocity on padded nodes, explicitly resetting them prevents numerical drift during integration.


In [8]:
def flow_loss_masked(v_pred, v_star, mask):
    """Mean-squared error over real atoms only."""
    err = (v_pred - v_star) ** 2
    err = err * mask[..., None]
    denom = mask.sum() * 3.0 + 1e-8
    return err.sum() / denom


@torch.no_grad()
def masked_rmsd(A, B, mask):
    """Per-sample Kabsch-aligned RMSD over real atoms only."""
    denom = mask.sum(dim=1, keepdim=True).clamp(min=1.0)
    A_center = (A * mask[..., None]).sum(dim=1, keepdim=True) / denom[..., None]
    B_center = (B * mask[..., None]).sum(dim=1, keepdim=True) / denom[..., None]
    A_centered = (A - A_center) * mask[..., None]
    B_centered = (B - B_center) * mask[..., None]
    cov = A_centered.transpose(1, 2) @ B_centered
    U, _, Vh = torch.linalg.svd(cov)
    det = torch.sign(torch.linalg.det(U @ Vh))
    corr = torch.eye(3, device=A.device, dtype=A.dtype).unsqueeze(0).repeat(A.shape[0], 1, 1)
    corr[:, -1, -1] = det
    rot = U @ corr @ Vh
    A_aligned = (A_centered @ rot) * mask[..., None]
    diff2 = ((A_aligned - B_centered) ** 2).sum(dim=-1) * mask
    msd = diff2.sum(dim=1) / denom.squeeze(1)
    return torch.sqrt(msd)


@torch.no_grad()
def euler_solve_masked(model, x0, cond, steps=80):
    """Euler integration with padded atoms frozen in place."""
    x = x0.clone()
    mask = cond["mask"]
    dt = 1.0 / steps
    for k in range(steps):
        t = torch.full((x.shape[0],), k * dt, device=x.device, dtype=x.dtype)
        v = model(x, t, cond) * mask[..., None]
        x = x + dt * v
        x = x * mask[..., None] + x0 * (1 - mask[..., None])
    return x


@torch.no_grad()
def evaluate_rmsd(model, dl, device, steps=80, max_batches=None):
    model.eval()
    start_rmsds = []
    pred_rmsds = []

    for bi, batch in enumerate(dl):
        xR = batch["xR"].to(device)
        xP = batch["xP"].to(device)
        xT = batch["xT"].to(device)
        z = batch["z"].to(device)
        x0g = batch["x0_guess"].to(device)
        mask = batch["mask"].to(device)

        x0 = x0g
        cond = {"z": z, "xR_ctx": xR, "xP_ctx": xP, "mask": mask}
        x_pred = euler_solve_masked(model, x0, cond, steps=steps)

        start_rmsds.append(masked_rmsd(x0, xT, mask).cpu())
        pred_rmsds.append(masked_rmsd(x_pred, xT, mask).cpu())

        if max_batches is not None and (bi + 1) >= max_batches:
            break

    start_rmsds = torch.cat(start_rmsds)
    pred_rmsds = torch.cat(pred_rmsds)
    return {
        "start_mean": float(start_rmsds.mean()),
        "pred_mean": float(pred_rmsds.mean()),
        "start_median": float(start_rmsds.median()),
        "pred_median": float(pred_rmsds.median()),
        "improvement_mean": float((start_rmsds - pred_rmsds).mean()),
    }


loss = flow_loss_masked(v_pred, v_star, mask)
print("masked flow loss:", float(loss.detach().cpu()))


masked flow loss: 0.12666644155979156


## Training
This is the same high-level training loop as the baseline notebook, but every batch now includes a mask and the loss is normalized by the number of active atom coordinates.

In effect, masking lets us train on the full variable-size dataset while still using standard dense tensor operations.


In [9]:
def train_epochs(model, train_dl, val_dl, device, epochs=5, lr=1e-4, ode_steps_eval=40):
    opt = torch.optim.Adam(model.parameters(), lr=lr)

    for ep in range(1, epochs + 1):
        model.train()
        running = 0.0
        nb = 0

        for batch in train_dl:
            x_t, t, cond, v_star, mask = make_flow_batch_padded(batch, device)
            opt.zero_grad()
            v_pred = model(x_t, t, cond)
            loss = flow_loss_masked(v_pred, v_star, mask)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            running += float(loss.detach().cpu())
            nb += 1

        metrics = evaluate_rmsd(model, val_dl, device, steps=ode_steps_eval, max_batches=100)
        print(
            f"epoch {ep} | train_loss {running / nb:.4f} | "
            f"val start {metrics['start_mean']:.4f} | "
            f"val pred {metrics['pred_mean']:.4f} | "
            f"imp {metrics['improvement_mean']:.4f}"
        )


model = FlowEGNN(h_dim=128, n_layers=3, use_concat_ctx=True).to(device)
train_epochs(model, train_dl, val_dl, device, epochs=5, lr=1e-4, ode_steps_eval=40)


epoch 1 | train_loss 0.0653 | val start 0.3780 | val pred 0.3751 | imp 0.0029
epoch 2 | train_loss 0.0623 | val start 0.3780 | val pred 0.3724 | imp 0.0056
epoch 3 | train_loss 0.0608 | val start 0.3780 | val pred 0.3755 | imp 0.0025


KeyboardInterrupt: 

## Final Evaluation
The comparison below asks whether the learned flow improves on the initial TS guess.
Because the metric is masked, reactions with fewer atoms are not artificially favored or penalized by padding.


In [ ]:
metrics = evaluate_rmsd(model, val_dl, device, steps=80, max_batches=50)
metrics
